# /news 端点连通性测试 (Client REST API)

走本地 RIT Client，不是 DMA 服务器。**跑之前 RIT Client 必须已启动并登录。**

跟 DMA 的区别只有两点：认证用 `X-API-Key` 而不是 Basic auth；限流由 Client 缓冲，基本不会吃 429。端点和返回格式完全一样。

In [ ]:
import requests

API_ENDPOINT = "http://localhost:9999/v1"

# 如果 401，就是这个 key 跟 RIT Client 里配置的对不上。
# 在 RIT Client 里找到 API 设置，把真实的 key 填到这里。
API_KEY = "Rotman"

session = requests.Session()
session.headers.update({"X-API-Key": API_KEY})
print("endpoint:", API_ENDPOINT, "| key:", repr(API_KEY))

In [ ]:
# 1. 连通性 + 认证诊断。不要用 resp.json()，401 的返回体可能不是 JSON。
try:
    resp = session.get(f"{API_ENDPOINT}/case", timeout=5)
except Exception as e:
    print("连不上：", type(e).__name__)
    print("-> RIT Client 没启动，或没监听 9999")
else:
    print("HTTP", resp.status_code, resp.reason)
    print("响应体:", resp.text[:500] or "(空)")
    if resp.status_code == 200:
        print("\nOK ->", resp.json())
    elif resp.status_code == 401:
        print("\n401 = Client 在跑但拒绝了这个 key。依次检查：")
        print("  1. RIT Client 是否已经登录并连上了一个案例（没登录也会 401）")
        print("  2. RIT Client 的 API 设置里真实的 key 是什么，填回上一个 cell 的 API_KEY")
        print("  3. 该案例是否允许 API 访问")

In [ ]:
# 2. 第一次拉新闻，不带游标
resp = session.get(f"{API_ENDPOINT}/news", params={"limit": 20})
print(resp.status_code)
news = resp.json()
news

In [ ]:
# 3. 核对字段名是否为 news_id / period / tick / ticker / headline / body
if news:
    print(sorted(news[0].keys()))
    print(news[0])

In [ ]:
# 4. 增量拉取：只要 news_id 比游标大的
last_news_id = max(n["news_id"] for n in news) if news else 0
print("last_news_id =", last_news_id)

resp2 = session.get(f"{API_ENDPOINT}/news", params={"after": last_news_id, "limit": 20})
print(resp2.status_code)
resp2.json()

In [ ]:
# 5. 用真实公告文本验证波动率解析
from vol_strategy import parse_vol_from_news

for n in news:
    print(n.get("tick"), "|", n.get("headline"), "->", parse_vol_from_news(n))

In [ ]:
# 6. 核对 /securities 的字段名（build_signal_table 依赖 ticker/last/bid/ask/position）
sec = session.get(f"{API_ENDPOINT}/securities").json()
print(sorted(sec[0].keys()))
sec[:3]